In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim 

import torchvision 
from torchvision.datasets import CIFAR10

In [14]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image => scale(0,1) => normalize => (-1,1)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)



In [16]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset , batch_size=64)

# Build the CNN

In [32]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10),

        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [33]:
model = CNN()

In [34]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [35]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) #fp
        loss = criterion(output , labels) # loss fnx
        loss.backward() # bp
        optimizer.step() #update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss ={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss =1.3519133840832869
epoch=2/10 & loss =0.9225223276316358
epoch=3/10 & loss =0.7338276957078358
epoch=4/10 & loss =0.602658286004725
epoch=5/10 & loss =0.4986226540392317
epoch=6/10 & loss =0.4001007704516811
epoch=7/10 & loss =0.3169895518771218
epoch=8/10 & loss =0.23937468920522334
epoch=9/10 & loss =0.18087226554961003
epoch=10/10 & loss =0.14940268351741687


In [36]:
# Evaluate our CNN

correct_labels = 0
total_labels= 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100} * 100")
    

accuracy = 73.88 * 100
